In [1]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import quilter
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'

## 导出用于绘图的数据

In [2]:
import mudata
scdata = mudata.read("/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_hm_outs/scenic_pipe/Snakemake/scplusmdata.h5mu")
scdata 

/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is depreca

MuData object with n_obs × n_vars = 19409 × 263328
  uns:	'direct_e_regulon_metadata', 'extended_e_regulon_metadata'
  6 modalities
    scRNA_counts:	19409 x 9911
      obs:	'leiden', 'eco', 'species', 'sample'
      var:	'gene_ids', 'feature_types', 'genome'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap'
    scATAC_counts:	19409 x 252091
      obs:	'sample', 'cisTopic_log_nr_acc', 'eco', 'cisTopic_nr_acc', 'barcode_rank', 'pdf_values_for_tss_enrichment', 'tss_enrichment', 'barcode', 'unique_fragments_in_peaks_count', 'log10_total_fragments_in_peaks_count', 'total_fragments_in_peaks_count', 'total_fragments_count', 'species', 'log10_total_fragments_count', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'pdf_values_for_fraction_of_fragments_in_peaks', 'pdf_values_for_duplication_ratio', 'nucleosome_signal', 'sample_id', 'duplication_ratio', 'log10_unique_fragments_in_peaks_count', 'duplication_count', 'log10_unique_fragments_count', 'unique_fragments_count', 'leiden', 'fraction_of_fragments_in_peaks', 'pycisTopic_leiden_10_1'
      var:	'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'
      obsm:	'UMAP'
    direct_gene_based_AUC:	19409 x 203
    direct_region_based_AUC:	19409 x 203
    extended_gene_based_AUC:	19409 x 460
    extended_region_based_AUC:	19409 x 460

In [21]:
grn = scdata.uns['direct_e_regulon_metadata']
tf_list = ['Mef2c','Rfx3','Pou6f1','Nfia','Nfix']

idx = []
for ele in grn['eRegulon_name'].values:
    if ele.split('_')[-1] == '+/+':
        idx.append(True)
    else:
        idx.append(False)

grn = grn.loc[idx,:]

idx = grn['TF'].isin(tf_list)
grn = grn.loc[idx,:]
print(grn.shape) 

tg_list = grn['Gene'].unique()
print(len(tg_list))
re_list = grn['Region'].unique()
print(len(re_list))

(734, 15)
374
634


### 根据OTOF阈值过滤网络

In [22]:
idx = grn['Gene'] == 'Otof'
delta = grn.loc[idx,'importance_TF2G'].min()

idx = grn['importance_TF2G'] >= delta 
print(idx.sum(), grn.shape[0])
grn = grn.sort_values('importance_TF2G', ascending=False)
print(grn.iloc[:400:,]['Gene'].nunique(), grn.iloc[:400,:]['Region'].nunique())
print(grn.iloc[:400]['TF'].value_counts())

grn_1 = grn.iloc[:400,:]
grn_2 = grn.loc[grn['TF'] == 'Pou6f1'].iloc[:20,:]
grn = pd.concat([grn_1, grn_2])



tg_list = grn['Gene'].unique()
print(len(tg_list))
re_list = grn['Region'].unique()
print(len(re_list))

175 734
188 357
Rfx3     180
Nfia     105
Nfix      91
Mef2c     24
Name: TF, dtype: int64
198
372


### 获取公共的 re的logFC

In [23]:
atac = scdata['scATAC_counts'][:,re_list]

idx = atac.obs.leiden == '0'

g1 = atac[idx,:].X.mean(0)
g2 = atac[~idx,:].X.mean(0) 

fc = np.log2(g1 + 1e-2/g2+1e-2)
re_dic = {}
for i,ele in enumerate(re_list):
    re_dic[ele] = fc[i]


import json
with open('re_dic.json', 'w') as f:
    json.dump(re_dic, f)

### 获取公共tg 的 logFC

In [24]:
rna = scdata['scRNA_counts'][:,tg_list]

idx = rna.obs.leiden == '0'

g1 = np.array(rna[idx,:].X.mean(0)).reshape(-1)
g2 = np.array(rna[~idx,:].X.mean(0)).reshape(-1)

fc = np.log2(g1 + 1e-2/g2+1e-2)
tg_dic = {}
for i,ele in enumerate(tg_list):
    tg_dic[ele] = fc[i].astype(np.float64)


import json
with open('tg_dic.json', 'w') as f:
    json.dump(tg_dic, f)

### 获取公共的tf 的logfc

In [25]:
rna = scdata['scRNA_counts'][:,tf_list]

idx = rna.obs.leiden == '0'

g1 = np.array(rna[idx,:].X.mean(0)).reshape(-1)
g2 = np.array(rna[~idx,:].X.mean(0)).reshape(-1)

fc = np.log2(g1 + 1e-2/g2+1e-2)
tf_dic = {}
for i,ele in enumerate(tf_list):
    tf_dic[ele] = fc[i].astype(np.float64)


import json
with open('tf_dic.json', 'w') as f:
    json.dump(tf_dic, f)

### 保存TF RE TG 信息

In [26]:
reg_list = []
for i in range(grn.shape[0]):
    re = grn['Region'].iloc[i]
    tg = grn['Gene'].iloc[i]
    tf = grn['TF'].iloc[i]
    reg_list.append(f'{tf}|{re}|{tg}')

with open('reg.txt', 'w') as f:
    for line in reg_list:
        f.write(line + '\n')

## 尝试复现scenicplus 给出的GRN结果

In [ ]:
out_dir = 'grn_plot'
import os 
os.makedirs(out_dir, exist_ok=True)

#Load scplus_obj
import dill
f_scplus_obj = '/staging/leuven/stg_00002/lcb/cbravo/Multiomics_pipeline/analysis/DPCL/scenicplus_final_autoreg/grnboost/scplus_obj.pkl'
scplus_obj = dill.load(open(f_scplus_obj, 'rb'))

#calculate triplet score and TF-to-region score
from mudata import AnnData
import mudata

adata_max_rank = mudata.read('/staging/leuven/stg_00002/lcb/sdewin/PhD/papers/SCENICPLUS_BravoDeWinter_etal_2022/revisions_nat_meth/triplet_score/DPCL_adata_max_rank.h5ad')
df_max_rank = adata_max_rank.to_df()

eRegulon_metadata_key = 'eRegulon_metadata'
TF_region_iter = scplus_obj.uns[eRegulon_metadata_key][['TF', 'Region']].to_numpy()
TF_to_region_score = [df_max_rank.loc[region, TF] for TF, region in TF_region_iter]
scplus_obj.uns[eRegulon_metadata_key]['TF_to_region_max_rank'] = TF_to_region_score

from scenicplus.triplet_score import _rank_scores_and_assign_random_ranking_in_range_for_ties, _calculate_cross_species_rank_ratio_with_order_statistics
TF2G_score_key = 'TF2G_importance'
R2G_score_key = 'R2G_importance'
TF2R_score_key = 'TF_to_region_max_rank'
TF2G_score = scplus_obj.uns[eRegulon_metadata_key][TF2G_score_key].to_numpy()
R2G_score = scplus_obj.uns[eRegulon_metadata_key][R2G_score_key].to_numpy()
TF2R_score = scplus_obj.uns[eRegulon_metadata_key][TF2R_score_key].to_numpy()
#rank the scores
TF2G_rank = _rank_scores_and_assign_random_ranking_in_range_for_ties(TF2G_score)
R2G_rank = _rank_scores_and_assign_random_ranking_in_range_for_ties(R2G_score)
TF2R_rank = _rank_scores_and_assign_random_ranking_in_range_for_ties(-TF2R_score) #negate because lower score is better

import numpy as np
TF2G_rank_ratio = (TF2G_rank.astype(np.float64) + 1) / TF2G_rank.shape[0]
R2G_rank_ratio = (R2G_rank.astype(np.float64) + 1) / R2G_rank.shape[0]
TF2R_rank_ratio = (TF2R_rank.astype(np.float64) + 1) / TF2R_rank.shape[0]

rank_ratios = np.array([TF2G_rank_ratio, R2G_rank_ratio, TF2R_rank_ratio])
aggregated_rank = np.zeros((rank_ratios.shape[1],), dtype = np.float64)
for i in range(rank_ratios.shape[1]):
    aggregated_rank[i] = _calculate_cross_species_rank_ratio_with_order_statistics(rank_ratios[:, i])

scplus_obj.uns[eRegulon_metadata_key]['triplet_score'] = aggregated_rank.argsort().argsort()

In [3]:
import mudata
scplus_mdata = mudata.read("/home/rsun@ZHANGroup.local/sly_data/grn_momics/scenic_hm_outs/scenic_pipe/Snakemake/scplusmdata.h5mu")
scplus_mdata 

/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is depreca

MuData object with n_obs × n_vars = 19409 × 263328
  uns:	'direct_e_regulon_metadata', 'extended_e_regulon_metadata'
  6 modalities
    scRNA_counts:	19409 x 9911
      obs:	'leiden', 'eco', 'species', 'sample'
      var:	'gene_ids', 'feature_types', 'genome'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap'
    scATAC_counts:	19409 x 252091
      obs:	'sample', 'cisTopic_log_nr_acc', 'eco', 'cisTopic_nr_acc', 'barcode_rank', 'pdf_values_for_tss_enrichment', 'tss_enrichment', 'barcode', 'unique_fragments_in_peaks_count', 'log10_total_fragments_in_peaks_count', 'total_fragments_in_peaks_count', 'total_fragments_count', 'species', 'log10_total_fragments_count', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'pdf_values_for_fraction_of_fragments_in_peaks', 'pdf_values_for_duplication_ratio', 'nucleosome_signal', 'sample_id', 'duplication_ratio', 'log10_unique_fragments_in_peaks_count', 'duplication_count', 'log10_unique_fragments_count', 'unique_fragments_count', 'leiden', 'fraction_of_fragments_in_peaks', 'pycisTopic_leiden_10_1'
      var:	'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'
      obsm:	'UMAP'
    direct_gene_based_AUC:	19409 x 203
    direct_region_based_AUC:	19409 x 203
    extended_gene_based_AUC:	19409 x 460
    extended_region_based_AUC:	19409 x 460

In [12]:
er_metadata = scplus_mdata.uns['direct_e_regulon_metadata']
nx_tables = {}
nx_tables['Edge'] = {}
nx_tables['Node'] = {}


In [ ]:
# Generate edge tables
r2g_columns = [x for x in er_metadata.columns if 'R2G' in x]
tf2g_columns = [x for x in er_metadata.columns if 'TF2G' in x]

nx_tables['Edge']['TF2R'] = er_metadata[er_metadata.columns.difference(
    r2g_columns + tf2g_columns)].drop('Gene', axis=1).drop_duplicates()
nx_tables['Edge']['TF2R'] = nx_tables['Edge']['TF2R'][['TF', 'Region'] +
                                                        nx_tables['Edge']['TF2R'].columns.difference(['TF', 'Region']).tolist()]
nx_tables['Edge']['R2G'] = er_metadata[er_metadata.columns.difference(
    tf2g_columns)].drop('TF', axis=1).drop_duplicates()
nx_tables['Edge']['R2G'] = nx_tables['Edge']['R2G'][['Region', 'Gene'] +
                                                    nx_tables['Edge']['R2G'].columns.difference(['Region', 'Gene']).tolist()]
nx_tables['Edge']['TF2G'] = er_metadata[er_metadata.columns.difference(
    r2g_columns)].drop('Region', axis=1).drop_duplicates()
nx_tables['Edge']['TF2G'] = nx_tables['Edge']['TF2G'][['TF', 'Gene'] +
                                                        nx_tables['Edge']['TF2G'].columns.difference(['TF', 'Gene']).tolist()]

In [23]:
# Generate node tables
tfs = list(set(er_metadata['TF']))
nx_tables['Node']['TF'] = pd.DataFrame(
    'TF', index=tfs, columns=['Node_type'])
nx_tables['Node']['TF']['TF'] = tfs
genes = list(set(er_metadata['Gene']))
genes = [x for x in genes if x not in tfs]
nx_tables['Node']['Gene'] = pd.DataFrame(
    'Gene', index=genes, columns=['Node_type'])
nx_tables['Node']['Gene']['Gene'] = genes
regions = list(set(er_metadata['Region']))
nx_tables['Node']['Region'] = pd.DataFrame(
    'Region', index=regions, columns=['Node_type'])
nx_tables['Node']['Region']['Region'] = regions

In [25]:
nx_tables['Node']['Gene']

,Node_type,Gene
Tcp11l2,Gene,Tcp11l2
Cog6,Gene,Cog6
Hectd1,Gene,Hectd1
Slc22a3,Gene,Slc22a3
Acp5,Gene,Acp5
...,...,...
Fut8,Gene,Fut8
Ilvbl,Gene,Ilvbl
Med10,Gene,Med10
Ccdc61,Gene,Ccdc61
